In [1]:
# Install required libraries
!pip install -q transformers torch scikit-learn

# Import libraries
from transformers import BertTokenizer, BertModel
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from sklearn.linear_model import LogisticRegression
import torch

# --------------------------------------------------
# 1. BERT - Sentence Embeddings
# --------------------------------------------------

# Load BERT tokenizer and model
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertModel.from_pretrained("bert-base-uncased")

# Example sentences
sentences = [
    "Artificial intelligence is useful in education.",
    "Students use computers for learning."
]

# Convert sentences into tokens
inputs = bert_tokenizer(
    sentences,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

# Generate BERT representations
with torch.no_grad():
    outputs = bert_model(**inputs)

# Take the [CLS] representation
embeddings = outputs.last_hidden_state[:, 0, :]

print("Sentence Embedding Shape:", embeddings.shape)
print("Each sentence is represented by a 768-dimensional vector.")


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6110.52it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can b

Sentence Embedding Shape: torch.Size([2, 768])
Each sentence is represented by a 768-dimensional vector.


In [2]:


# --------------------------------------------------
# 2. BERT - Simple Classification
# --------------------------------------------------

train_sentences = [
    "The student attended the class.",
    "The teacher explained the lesson.",
    "The computer stopped working.",
    "The software has an error."
]

# 1 = Education, 0 = Technology Problem
labels = [1, 1, 0, 0]

# Convert training sentences into BERT embeddings
train_inputs = bert_tokenizer(
    train_sentences,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

with torch.no_grad():
    train_outputs = bert_model(**train_inputs)

train_embeddings = train_outputs.last_hidden_state[:, 0, :]

# Train a simple classifier
classifier = LogisticRegression()
classifier.fit(train_embeddings.numpy(), labels)

# Test new sentences
test_sentences = [
    "The professor conducted a class.",
    "My computer is not working."
]

test_inputs = bert_tokenizer(
    test_sentences,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

with torch.no_grad():
    test_outputs = bert_model(**test_inputs)

test_embeddings = test_outputs.last_hidden_state[:, 0, :]

predictions = classifier.predict(test_embeddings.numpy())

print("\nClassification Results:")

for sentence, prediction in zip(test_sentences, predictions):
    category = "Education" if prediction == 1 else "Technology Problem"
    print(sentence, "→", category)




Classification Results:
The professor conducted a class. → Education
My computer is not working. → Technology Problem


In [3]:

# --------------------------------------------------
# 3. GPT-2 - Text Generation
# --------------------------------------------------

gpt_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
gpt_model = GPT2LMHeadModel.from_pretrained("gpt2")

# GPT-2 requires a padding token
gpt_tokenizer.pad_token = gpt_tokenizer.eos_token

prompt = "Artificial intelligence can help students"

# Convert prompt into tokens
input_ids = gpt_tokenizer.encode(
    prompt,
    return_tensors="pt"
)

# Generate text
with torch.no_grad():
    generated = gpt_model.generate(
        input_ids,
        max_length=40,
        do_sample=True,
        temperature=0.7
    )

# Display generated text
text = gpt_tokenizer.decode(
    generated[0],
    skip_special_tokens=True
)

print("\nGPT-2 Generated Text:")
print(text)


# --------------------------------------------------
# 4. BERT vs GPT
# --------------------------------------------------

print("\nBERT vs GPT")

print("BERT → Understands text using an Encoder")
print("GPT  → Generates text using a Decoder")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 5953.08it/s]



GPT-2 Generated Text:
Artificial intelligence can help students improve their English language skills.

A team of researchers from the University of Cambridge have published their findings in the journal Science.

They found that students who had

BERT vs GPT
BERT → Understands text using an Encoder
GPT  → Generates text using a Decoder
